In [26]:
import pandas as pd
import math

# Cargar el archivo principal
file_path = 'C:/Users/VALE/Desktop/Tesis/Tesis Pedro Palominos/Código python/Modelo_info_historica/archivos_estaciones_elegidas20/todos_datos_elegidas.xlsx'
sheet_name = "BDE18a22dic"
df = pd.read_excel(file_path, sheet_name=sheet_name)

# Asegurarse de que la columna "Hora" sea de tipo datetime
df['Hora'] = pd.to_datetime(df['Hora'], format='%d-%m-%Y %H:%M:%S')

# Extraer la fecha y la hora por separado
df['Fecha'] = df['Hora'].dt.date
df['Solo_Hora'] = df['Hora'].dt.strftime('%H:%M')

# Fecha y hora deseada
fecha_deseada = '2024-12-08'
hora_deseada = '18:00'
fecha_deseada = pd.to_datetime(fecha_deseada).date()

# Filtrar los datos por la fecha y hora deseada
resultado = df[(df['Fecha'] == fecha_deseada) & (df['Solo_Hora'] == hora_deseada)][['Estacion', 'Bikes disponibles']]

# Cargar el archivo de promedios
promedio_file_path = 'C:/Users/VALE/Desktop/Tesis/Tesis Pedro Palominos/Código python/Modelo_info_historica/archivos_estaciones_elegidas20/promedio_bikes_por_estacion_20.xlsx'
promedio_df = pd.read_excel(promedio_file_path)

# Verificar que el archivo de promedios tenga las columnas necesarias
if 'Estacion' not in promedio_df.columns or 'Promedio Bikes' not in promedio_df.columns:
    raise ValueError("El archivo de promedios no tiene las columnas esperadas ('Estacion', 'Promedio Bikes').")

# Crear una lista completa de estaciones desde el archivo de promedios
estaciones_completas = promedio_df['Estacion'].unique()

# Iterar por todas las estaciones y completar valores faltantes
valores_finales = []
for estacion in estaciones_completas:
    # Buscar en el resultado filtrado
    fila = resultado[resultado['Estacion'] == estacion]
    if fila.empty:  # Si no se encuentra, usar el promedio
        promedio = promedio_df.loc[promedio_df['Estacion'] == estacion, 'Promedio Bikes'].values
        if len(promedio) > 0:
            valores_finales.append({'Estacion': estacion, 'Bikes disponibles': math.ceil(promedio[0])})
        else:
            print(f"No se encontró promedio para la estación {estacion}.")
    else:  # Si se encuentra, usar el valor existente
        valores_finales.append({'Estacion': estacion, 'Bikes disponibles': fila['Bikes disponibles'].values[0]})

# Crear un DataFrame con los valores finales
resultado_final = pd.DataFrame(valores_finales)

# Ordenar las estaciones alfabéticamente
resultado_final = resultado_final.sort_values(by='Estacion', key=lambda col: col.str.lower())

# Agregar una columna con la fecha y hora seleccionada
resultado_final['Fecha_Hora'] = f"{fecha_deseada} {hora_deseada}"

# Guardar el resultado en un nuevo archivo Excel
output_file = f"bikes_disponibles_{fecha_deseada}_{hora_deseada.replace(':', '-')}.xlsx"
resultado_final.to_excel(output_file, index=False)
print(f"Los datos completos se han guardado en: {output_file}")


Los datos completos se han guardado en: bikes_disponibles_2024-12-08_18-00.xlsx
